In [2]:
import rasterio
from rasterio.features import rasterize
import geopandas
import numpy as np


In [4]:
# Import the land use shapefile
# C:\SoilGrids\cobertura_vegetal\landuse_clipped_MB2.shp
landuse_shapefile_path = r"C:\SoilGrids\cobertura_vegetal\landuse_clipped_MB2.shp"
landuse_gdf = geopandas.read_file(landuse_shapefile_path)

In [6]:

# Print the unique values of the 'ctn1' and 'ctn2' columns
print("Unique values in 'ctn1' column:")
print(landuse_gdf['ctn1'].unique())
print("\nUnique values in 'ctn2' column:")
print(landuse_gdf['ctn2'].unique())

# Create a mapping from unique ctn2 values to numeric codes
unique_ctn2 = landuse_gdf['ctn2'].unique()
ctn2_to_code = {value: idx + 1 for idx, value in enumerate(unique_ctn2)}

# Print the mapping
print("\nctn2 to numeric code mapping:")
for value, code in ctn2_to_code.items():
    print(f"  {code}: {value}")

# Create a new column with the numeric codes
landuse_gdf['ctn2_code'] = landuse_gdf['ctn2'].map(ctn2_to_code)

print("\nNew column 'ctn2_code' added to the geodataframe.")

# Save the updated geodataframe back to the shapefile
landuse_gdf.to_file(landuse_shapefile_path)
print(f"Shapefile saved with new 'ctn2_code' column: {landuse_shapefile_path}")



Unique values in 'ctn1' column:
['OTRAS TIERRAS' 'BOSQUE' 'TIERRA AGROPECUARIA'
 'VEGETACION ARBUSTIVA Y HERBACEA' 'ZONA ANTROPICA' 'CUERPO DE AGUA']

Unique values in 'ctn2' column:
['AREA SIN COBERTURA VEGETAL' 'BOSQUE NATIVO' 'PASTIZAL'
 'MOSAICO AGROPECUARIO' 'VEGETACION ARBUSTIVA Y HERBACEA'
 'INFRAESTRUCTURA' 'PARAMO' 'CUERPO DE AGUA NATURAL' 'PLANTACION FORESTAL'
 'AREA POBLADA' 'CUERPO DE AGUA ARTIFICIAL']

ctn2 to numeric code mapping:
  1: AREA SIN COBERTURA VEGETAL
  2: BOSQUE NATIVO
  3: PASTIZAL
  4: MOSAICO AGROPECUARIO
  5: VEGETACION ARBUSTIVA Y HERBACEA
  6: INFRAESTRUCTURA
  7: PARAMO
  8: CUERPO DE AGUA NATURAL
  9: PLANTACION FORESTAL
  10: AREA POBLADA
  11: CUERPO DE AGUA ARTIFICIAL

New column 'ctn2_code' added to the geodataframe.
Shapefile saved with new 'ctn2_code' column: C:\SoilGrids\cobertura_vegetal\landuse_clipped_MB2.shp


rasterize landuse

In [5]:
# Rasterize the landuse using ctn2_code, matching the DEM resolution and CRS
dem_path = r"C:\SoilGrids\inputs_swat\dem_mazar_reprojected.tif"
output_raster_path = r"C:\SoilGrids\cobertura_vegetal\landuse_rasterized_ctn2.tif"

# Read the DEM to get the reference CRS, transform, and shape
with rasterio.open(dem_path) as dem:
    dem_crs = dem.crs
    dem_transform = dem.transform
    dem_shape = (dem.height, dem.width)
    dem_bounds = dem.bounds
    print(f"\nDEM CRS: {dem_crs}")
    print(f"DEM resolution: {dem_transform[0]} x {-dem_transform[4]}")
    print(f"DEM shape: {dem_shape}")

# Reproject the landuse geodataframe to match the DEM CRS if necessary
if landuse_gdf.crs != dem_crs:
    print(f"\nReprojecting landuse from {landuse_gdf.crs} to {dem_crs}...")
    landuse_gdf = landuse_gdf.to_crs(dem_crs)

# Create geometry-value pairs for rasterization
shapes = [(geom, value) for geom, value in zip(landuse_gdf.geometry, landuse_gdf['ctn2_code'])]

# Rasterize the landuse shapefile
landuse_raster = rasterize(
    shapes=shapes,
    out_shape=dem_shape,
    transform=dem_transform,
    fill=0,  # NoData value for areas outside polygons
    dtype=np.int16
)

# Save the rasterized landuse
with rasterio.open(
    output_raster_path,
    'w',
    driver='GTiff',
    height=dem_shape[0],
    width=dem_shape[1],
    count=1,
    dtype=np.int16,
    crs=dem_crs,
    transform=dem_transform,
    nodata=0
) as dst:
    dst.write(landuse_raster, 1)

print(f"\nLanduse rasterized and saved to: {output_raster_path}")


DEM CRS: EPSG:32717
DEM resolution: 30.80189573595455 x 30.80189573595455
DEM shape: (3042, 2864)

Landuse rasterized and saved to: C:\SoilGrids\cobertura_vegetal\landuse_rasterized_ctn2.tif
